# Conversational **Hybrid RAG** with Pinecone + Gemini
### Dense (meaning) + Sparse (keywords) search in a single index

This notebook builds a RAG system that searches with **both** a *dense* vector (semantic meaning) and a *sparse* vector (keyword/BM(BestMatch-25) at the same time — Pinecone stores both and blends them with a single knob called **alpha**.

## Steps
1. What is RAG, and why **hybrid**?
2. Load the document
3. Chunk the text
4. **Dense** embeddings (meaning)
5. **Sparse** embeddings (BM25 keywords)
6. Create a Pinecone index
7. Upsert hybrid (dense + sparse) vectors
8. **Hybrid search** with the `alpha` knob
9. Generate the answer with Gemini
10. Conversational RAG (with memory)

## 1. What is RAG, and why hybrid?

An LLM only knows its training data. **RAG** (Retrieval-Augmented Generation) fixes that: we **retrieve** relevant passages from our own documents, **add** them to the prompt, and let the LLM **generate** an answer grounded in them.

`question → search documents → put best passages in prompt → LLM answers`

The quality of RAG depends almost entirely on the **search** step. There are two kinds of search, and each is good at what the other is bad at:

| | **Dense** (vector / semantic) | **Sparse** (BM25 / keyword) |
|---|---|---|
| Matches on | *Meaning* | *Exact words* |
| Good at | synonyms, paraphrases, concepts | names, codes, acronyms, numbers |
| Example | "car" finds "automobile" | finds the exact term "ERP-2008" |

**Hybrid search runs both at once.** Pinecone makes this easy: each item gets a **dense vector** *and* a **sparse vector** in the same index, and at query time a single `alpha` value decides how much each one counts.

In [13]:
#!pip install ipykernel pinecone pinecone-text sentence-transformers PyPDF2 google-genai python-dotenv

In [1]:
import os
import time
import PyPDF2
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)  # Load environment variables from .env file

# ---- API keys (for a demo only; in production load these from env vars) ----
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY")

## 2. Load the document

First we read the raw text out of our file. We support `.txt` and `.pdf` here.

In [2]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()


def read_pdf_file(file_path):
    text = ""
    with open(file_path, 'rb') as f:
        for page in PyPDF2.PdfReader(f).pages:
            text += (page.extract_text() or "") + "\n"
    return text


def read_document(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.txt':
        return read_text_file(file_path)
    if ext == '.pdf':
        return read_pdf_file(file_path)
    raise ValueError(f"Unsupported file format: {ext}")


DOC_PATH = r"H:\01_Training\ContentSlides_AgenticAI\Code\Deployment_RAG\docs\ERP-2008-chapter4.pdf"
text = read_document(DOC_PATH)
print(f"Loaded {len(text):,} characters")
print(text[:400], "...")

Loaded 46,873 characters
97CHAPTER 4
The Importance of Health and 
Health Care
The American health care system is an engine for innovation that develops 
and broadly disseminates advanced, life-enhancing treatments and offers 
a wide set of choices for consumers of health care. The current health care system provides enormous benefits, but there are substantial opportunities for reforms that would reduce costs, increase a ...


## 3. Chunk the text

We split the document into small passages (**chunks**). Search and the LLM both work better on focused passages than on one giant blob of text.

In [3]:
def split_text(text, chunk_size=500):
    """Split text into ~chunk_size-charac chunks made of whole sentences."""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('. ') if s.strip()]
    chunks, current, size = [], [], 0
    for s in sentences:
        if not s.endswith('.'):
            s += '.'
        if size + len(s) > chunk_size and current:
            chunks.append(' '.join(current))
            current, size = [], 0
        current.append(s)
        size += len(s)
    if current:
        chunks.append(' '.join(current))
    return chunks


chunks = split_text(text)
print(f"Created {len(chunks)} chunks")
print("\nExample chunk:\n", chunks[2])

Created 115 chunks

Example chunk:
 But because health care financing and delivery are often inefficient, there are opportunities to advance health and access to health care services without further growth in spending. To improve the efficiency of health care financing and delivery, the Administration has pursued policies that would increase incentives for individuals to purchase consumer-directed health insurance plans.


## 4. Dense embeddings (meaning)

A **dense embedding** turns a piece of text into a list of numbers (a *vector*) that captures its *meaning*. Texts with similar meaning get similar vectors. We use a small, free, local model `all-MiniLM-L6-v2` which produces **384-dimensional** vectors.

In [4]:
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer("all-MiniLM-L6-v2")
DIM = dense_model.get_embedding_dimension()


def embed_dense(texts):
    """texts: str or list[str] -> list[list[float]]"""
    if isinstance(texts, str):
        texts = [texts]
    return dense_model.encode(texts).tolist()


example = embed_dense("What is the American health system?")[0]
print(f"Dense vector dimension: {DIM}")
print("First 8 numbers:", [round(x, 3) for x in example[:8]])

h:\01_Training\ContentSlides_AgenticAI\Code\all_envs\HybridRAG_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 888.55it/s]


Dense vector dimension: 384
First 8 numbers: [0.022, 0.031, -0.092, -0.012, -0.026, 0.028, 0.003, 0.049]


## 5. Sparse embeddings (BM25(BestMatching) keywords)

A **sparse embedding** is the keyword side. It is a *huge* vector where most values are 0, and only the positions for the words actually present have a value. The value is a **BM25 weight** — higher for important words.

**BM25** decides word importance using three ideas:
- **Term frequency:** a word that appears more often is more relevant — but with diminishing returns.
- **Inverse document frequency (IDF):** rare words (like "longevity") matter more than common ones (like "the").
- **Length normalization:** long chunks don't get an unfair advantage just for having more words.

We use Pinecone's `BM25Encoder`. We **fit** it on our chunks (so it learns the word statistics), then encode documents and queries. Note the two different methods:
- `encode_documents(...)` — for the chunks we store
- `encode_queries(...)` — for the user's question

In [5]:
from pinecone_text.sparse import BM25Encoder

bm25 = BM25Encoder()
bm25.fit(chunks)   # learn word statistics from our document

sample = bm25.encode_queries("What is the demand for health?")
print("A sparse vector is just indices + values:")
print("  indices:", sample['indices'][:8], "...")
print("  values :", [round(v, 3) for v in sample['values'][:8]], "...")
print(f"  ({len(sample['indices'])} non-zero terms — everything else is 0)")

100%|██████████| 115/115 [00:00<00:00, 116.96it/s]

A sparse vector is just indices + values:
  indices: [1504846510, 1357846105] ...
  values : [0.912, 0.088] ...
  (2 non-zero terms — everything else is 0)


## 6. Create a Pinecone index

To store dense + sparse together, the index **must** use the `dotproduct` metric — this is what lets Pinecone combine the two vector types in one similarity score. We set the dimension to match our dense model (384).

In [6]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "hybrid-rag"

# (re)create the index
if pc.has_index(INDEX_NAME):
    pc.delete_index(INDEX_NAME)

pc.create_index(
    name=INDEX_NAME,
    dimension=DIM,
    metric="dotproduct",                       # REQUIRED for sparse-dense hybrid
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

# wait until it is ready
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

index = pc.Index(INDEX_NAME)
print("Index ready:", INDEX_NAME)

Index ready: hybrid-rag


## 7. Upsert hybrid vectors

Now we push each chunk into Pinecone. Every record carries:
- `values` — the **dense** vector (meaning)
- `sparse_values` — the **sparse** vector (keywords)
- `metadata` — the original text + source, so we can read it back after search

In [7]:
def upsert_chunks(chunks, source, batch_size=50):
    dense_vecs = embed_dense(chunks)               # list of dense vectors
    sparse_vecs = bm25.encode_documents(chunks)    # list of {indices, values}

    vectors = []
    for i, chunk in enumerate(chunks):
        vectors.append({
            "id": f"{source}_chunk_{i}",
            "values": dense_vecs[i],
            "sparse_values": sparse_vecs[i],
            "metadata": {"text": chunk, "source": source, "chunk": i},
        })

    for i in range(0, len(vectors), batch_size):
        index.upsert(vectors=vectors[i:i + batch_size])
    print(f"Upserted {len(vectors)} chunks")


upsert_chunks(chunks, source="ERP-2008-chapter4.pdf")
time.sleep(5)  # give the index a moment to index the new vectors
print(index.describe_index_stats())

Upserted 115 chunks
DescribeIndexStatsResponse(dimension=384, total_vector_count=115, metric='dotproduct', namespaces=1)


## 8. Hybrid search with the `alpha` knob

At query time we build **both** vectors for the question, then scale them by `alpha` before searching:

- `alpha = 1.0` → **pure dense** (meaning only)
- `alpha = 0.0` → **pure sparse** (keywords only)
- `alpha = 0.5` → an even blend (good default)

The trick is simple: multiply the dense values by `alpha` and the sparse values by `1 - alpha`. Because the index uses dot-product, scaling the vectors directly scales how much each side contributes to the final score.

In [8]:
def hybrid_scale(dense, sparse, alpha):
    """Weight the two vectors. alpha=1 dense-only, alpha=0 sparse-only."""
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")
    scaled_dense = [v * alpha for v in dense]
    scaled_sparse = {
        "indices": sparse["indices"],
        "values": [v * (1 - alpha) for v in sparse["values"]],
    }
    return scaled_dense, scaled_sparse


def hybrid_search(query, top_k=4, alpha=0.5):
    dense = embed_dense(query)[0]
    sparse = bm25.encode_queries(query)
    hdense, hsparse = hybrid_scale(dense, sparse, alpha)

    res = index.query(
        vector=hdense,
        sparse_vector=hsparse,
        top_k=top_k,
        include_metadata=True,
    )
    return res["matches"]


def show(matches, title):
    print(f"\n{title}\n" + "-" * 60)
    for m in matches:
        print(f"score={m['score']:.4f}  (chunk {m['metadata']['chunk']})")
        print("  ", m['metadata']['text'][:150], "...\n")

In [9]:
query = "What is the demand for health?"

show(hybrid_search(query, alpha=1.0), "alpha=1.0  -> DENSE only (meaning)")
show(hybrid_search(query, alpha=0.0), "alpha=0.0  -> SPARSE only (keywords)")


alpha=1.0  -> DENSE only (meaning)
------------------------------------------------------------
score=0.7024  (chunk 6)
   chapter4.indd 97chapter4.indd   97 2/5/08 1:22:31 PM2/5/08   1:22:31 PM 98 | Economic Report of the PresidentHealth and the Demand for Health Care The ...

score=0.6356  (chunk 8)
   Demand for Health People demand health because of its role in facilitating and providing  happiness. Health can be defined along two dimensions: the l ...

score=0.5862  (chunk 7)
   For example, demand for an MP3 player is based on the enjoyment that an MP3 player brings to a consumer, but few would choose to get a laparoscopic ch ...

score=0.5762  (chunk 21)
   Medical technology may account for about one-half or more of real long-term  chapter4.indd 100chapter4.indd   100 2/5/08 1:22:32 PM2/5/08   1:22:32 PM ...


alpha=0.0  -> SPARSE only (keywords)
------------------------------------------------------------
score=0.6921  (chunk 6)
   chapter4.indd 97chapter4.indd   97 2/5/08 1:

In [11]:
show(hybrid_search(query, alpha=0.5), "alpha=0.5  -> HYBRID (both)")


alpha=0.5  -> HYBRID (both)
------------------------------------------------------------
score=0.6972  (chunk 6)
   chapter4.indd 97chapter4.indd   97 2/5/08 1:22:31 PM2/5/08   1:22:31 PM 98 | Economic Report of the PresidentHealth and the Demand for Health Care The ...

score=0.6274  (chunk 8)
   Demand for Health People demand health because of its role in facilitating and providing  happiness. Health can be defined along two dimensions: the l ...

score=0.6010  (chunk 7)
   For example, demand for an MP3 player is based on the enjoyment that an MP3 player brings to a consumer, but few would choose to get a laparoscopic ch ...

score=0.4404  (chunk 59)
   If a new product is only slightly more effective than an existing product, for example, it may be highly demanded even if it is priced well above exis ...



## 9. Generate the answer with Gemini

We take the retrieved chunks, paste them into the prompt as **context**, and ask Gemini to answer using only that context. The "only" instruction is what stops the model from making things up.

In [10]:
def get_prompt(query, context):
    return f"""Based ONLY on the following context, answer the question.
If the answer is not in the context, say "I cannot answer this based on the provided context."

Context:
{context}

Question: {query}

Answer:"""

In [11]:
get_prompt("What is the demand for health?", "this is my context chunks"  )

'Based ONLY on the following context, answer the question.\nIf the answer is not in the context, say "I cannot answer this based on the provided context."\n\nContext:\nthis is my context chunks\n\nQuestion: What is the demand for health?\n\nAnswer:'

In [12]:
genai_client = genai.Client(api_key=GOOGLE_API_KEY)


def get_prompt(query, context):
    return f"""Based ONLY on the following context, answer the question.
If the answer is not in the context, say "I cannot answer this based on the provided context."

Context:
{context}

Question: {query}

Answer:"""


def gemini_generate(prompt):
    resp = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return resp.text


def rag_query(query, top_k=4, alpha=0.5):
    matches = hybrid_search(query, top_k=top_k, alpha=alpha)
    context = "\n\n".join(m["metadata"]["text"] for m in matches)
    sources = [f"{m['metadata']['source']} (chunk {m['metadata']['chunk']})" for m in matches]
    answer = gemini_generate(get_prompt(query, context))
    return answer, sources

In [13]:
answer, sources = rag_query("What is the demand for health?", alpha=0.5)
print("Answer:\n", answer)
print("\nSources:")
for s in sources:
    print(" -", s)

Answer:
 People demand health because of its role in facilitating and providing happiness. Health can be defined along two dimensions: the length of life (longevity) and the quality of life.

Sources:
 - ERP-2008-chapter4.pdf (chunk 6)
 - ERP-2008-chapter4.pdf (chunk 8)
 - ERP-2008-chapter4.pdf (chunk 7)
 - ERP-2008-chapter4.pdf (chunk 59)


## 10. Conversational RAG (with memory)

For a chat, follow-up questions like *"how is it different?"* don't make sense on their own. So before searching, we ask the LLM to **rewrite** the follow-up into a standalone question using the chat history. Then we run normal hybrid RAG and remember the turn.

In [30]:
class ConversationalRAG:
    def __init__(self, top_k=4, alpha=0.5, max_history=5):
        self.history = []
        self.top_k = top_k
        self.alpha = alpha
        self.max_history = max_history

    def _standalone(self, query):
        if not self.history:
            return query
        hist = "\n".join(f"User: {u}\nAssistant: {a}" for u, a in self.history)
        prompt = (
            "Rewrite the follow-up question as a standalone question using the "
            "conversation history (resolve words like 'it' or 'that'). "
            "Output only the rewritten question.\n\n"
            f"History:\n{hist}\n\nFollow-up: {query}\nStandalone:"
        )
        return gemini_generate(prompt).strip()

    def chat(self, query):
        standalone = self._standalone(query)
        if standalone != query:
            print(f"[rewritten] {standalone}")
        answer, sources = rag_query(standalone, self.top_k, self.alpha)
        self.history.append((query, answer))
        self.history = self.history[-self.max_history:]
        return answer, sources

In [31]:
bot = ConversationalRAG(alpha=0.5)

for q in [
    "What is the demand for health?",
    "How is it different from the demand for other products?",  # 'it' = demand for health
]:
    print("=" * 70)
    print("USER:", q)
    ans, srcs = bot.chat(q)
    print("\nBOT :", ans)
    print("Sources:", ", ".join(srcs))
    print()

USER: What is the demand for health?

BOT : People demand health because of its role in facilitating and providing happiness. Health can be defined along two dimensions: the length of life (longevity) and the quality of life. A person derives value from the quality of life directly and indirectly: directly because one’s level of health affects the enjoyment of goods and leisure, and indirectly because one’s level of health enhances productivity.
Sources: ERP-2008-chapter4.pdf (chunk 6), ERP-2008-chapter4.pdf (chunk 8), ERP-2008-chapter4.pdf (chunk 7), ERP-2008-chapter4.pdf (chunk 59)

USER: How is it different from the demand for other products?
[rewritten] How is the demand for health different from the demand for other products?

BOT : The demand for health care is unlike the demand for most consumer products and services because while the desire for consumer products and services comes from direct consumption, the desire for health care is not derived directly from the consumption o

## Recap

```
  document → chunks ─┬─ dense embed (meaning)  ─┐
                     └─ BM25 sparse (keywords) ─┴─→ Pinecone (dotproduct index)

  question ─┬─ dense embed  ─┐  scale by alpha
           └─ BM25 sparse  ─┴─→ hybrid query → top-k chunks → Gemini → answer
```

- **Dense** = meaning, **Sparse (BM25)** = exact keywords.
- Pinecone stores both in one `dotproduct` index.
- **`alpha`** blends them: `1`=meaning only, `0`=keywords only, `0.5`=balanced.
- The conversational layer rewrites follow-ups into standalone questions before searching.

**Try next:** tune `alpha` per query type, add more documents, or `pip install` a re-ranker to reorder the top results.

## 11. Validate the RAG with RAGAS (LLM-as-a-judge, **no ground truth**)

In the real world you rarely have a "correct answer" for every question users ask — if you did, you wouldn't need the RAG. So we validate using metrics that need **only what the pipeline already produces**: the question, the retrieved chunks, and the generated answer. An **LLM (Gemini)** acts as the judge.

All three metrics are **reference-free** and on a **0–1 scale** (higher is better):

| Metric | Question it answers | Needs (no ground truth!) |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context (no hallucination)? | answer + contexts |
| **Answer Relevancy** | Does the answer actually address the question? | question + answer |
| **Context Precision** | Are the retrieved chunks relevant to answering? | question + answer + contexts |

> Note: RAGAS *also* offers reference-based metrics (`context_recall`, `answer_correctness`) — but those require hand-labelled ground truth, which defeats the purpose here. We deliberately skip them.

In [ ]:
#!pip install ragas langchain-google-genai langchain-huggingface datasets

In [ ]:
# Wrap our existing models so RAGAS can use them as the "judge".
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Gemini is the LLM judge; MiniLM (already downloaded above) handles embedding metrics.
# evaluator_llm = LangchainLLMWrapper(
#     ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY, temperature=0)
# )

evaluator_llm = LangchainLLMWrapper(genai_client.models.generate_content(model="gemini-2.5-flash",contents="evaluation of RAG"))


# evaluator_emb = LangchainEmbeddingsWrapper(
#     HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# )

evaluator_emb = LangchainEmbeddingsWrapper(dense_model)
print("RAGAS judge ready: Gemini (LLM) + MiniLM (embeddings)")

RAGAS judge ready: Gemini (LLM) + MiniLM (embeddings)


In [19]:
# Just the questions — NO ground-truth answers needed.
eval_questions = [
    "What is the demand for health?",
    "How is the demand for health care different from the demand for other consumer products?",
    "What share of long-term growth in health spending is attributed to medical technology?",
]


def rag_with_contexts(query, top_k=4, alpha=0.5):
    """Like rag_query, but also returns the raw context chunks RAGAS needs."""
    matches = hybrid_search(query, top_k=top_k, alpha=alpha)
    contexts = [m["metadata"]["text"] for m in matches]
    answer = gemini_generate(get_prompt(query, "\n\n".join(contexts)))
    return answer, contexts


# Run the hybrid RAG over each question to collect answers + retrieved contexts.
answers, contexts_list = [], []
for q in eval_questions:
    ans, ctxs = rag_with_contexts(q, alpha=0.5)
    answers.append(ans)
    contexts_list.append(ctxs)
    print(f"Q: {q}\nA: {ans[:120]}...\n")

Q: What is the demand for health?
A: People demand health because of its role in facilitating and providing happiness. Health can be defined along two dimens...

Q: How is the demand for health care different from the demand for other consumer products?
A: The demand for health care is unlike the demand for most consumer products and services because while the desire for con...

Q: What share of long-term growth in health spending is attributed to medical technology?
A: Medical technology may account for about one-half or more of real long-term health care spending growth....



In [24]:
# Assemble the dataset (no ground_truth column) and run RAGAS with reference-free metrics.
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.metrics import LLMContextPrecisionWithoutReference

# Context precision that judges relevance using the question + answer (NOT a reference).
context_precision = LLMContextPrecisionWithoutReference()

eval_dataset = Dataset.from_dict({
    "question": eval_questions,
    "answer":   answers,
    "contexts": contexts_list,
})

result = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=evaluator_llm,
    embeddings=evaluator_emb,
)

print("\n=== RAGAS scores (0-1, higher is better, no ground truth) ===")
print(result)

# Per-question breakdown as a table.
result.to_pandas()

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

### How to read the scores

All three came from the LLM judge using **only the pipeline's own outputs** — no answer key required:

- **Faithfulness** low → the model is **hallucinating** beyond the retrieved chunks. Tighten the prompt or retrieve better context.
- **Answer Relevancy** low → the answer wanders off-topic. Check the prompt wording.
- **Context Precision** low → retrieval is pulling **irrelevant chunks**. Tune `alpha`, lower `top_k`, or improve chunking.

This is exactly what you want when you have **no ground truth**: the judge tells you whether to fix **retrieval** (context precision) or **generation** (faithfulness / relevancy). Re-run with a different `alpha` and watch the scores move.

> The one thing reference-free metrics *can't* tell you is whether the answer is **factually complete** — for that you'd need ground truth (`context_recall` / `answer_correctness`). Reference-free catches hallucination and irrelevance, not "what the docs never retrieved."

In [23]:
# !pip install datasets
!pip install ragas

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
     ---------------------------------------- 0.0/40.9 kB ? eta -:--:--
     ------------------------------ --------- 30.7/40.9 kB 1.4 MB/s eta 0:00:01
     -------------------------------------- 40.9/40.9 kB 493.8 kB/s eta 0:00:00
  Using cached rich-14.3.4-py3-none-any.whl.metadata (18 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using c


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
